## Install Packages

In [1]:
!pip install evaluate trackio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 102.2 MB/s eta 0:00:00
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: gradio
    Found existing installation: gradio 5.50.0
    Uninstalling gradio-5.50.0:
      Successfully uninstalled gradio-5.50.0


## Imports

In [2]:
from datasets import DatasetDict, Value
import evaluate
from huggingface_hub import login
from kagglehub import KaggleDatasetAdapter, dataset_load
from kaggle_secrets import UserSecretsClient
import numpy as np
import os
from peft import get_peft_model, LoraConfig, TaskType
import random
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, set_seed, Trainer, TrainingArguments

## Set Seed

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)           # CPU
torch.cuda.manual_seed(SEED)      # Current GPU
torch.cuda.manual_seed_all(SEED)  # All GPUs (if multiple)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
set_seed(SEED)

## Configs

In [4]:
# General configurations
DATASET_PATH = "bingxuanchia/dsa4262-finetuning-exploration" # Either "regression" or "classification"
MODEL_NAME = "google-bert/bert-base-cased"

FINETUNING_METHOD = "classification"
FINETUNED_MODEL_NAME = f"heladepdet-bert-finetuned-{FINETUNING_METHOD}"
FINETUNED_MODEL_REPO_ID = f"chiabingxuan/{FINETUNED_MODEL_NAME}"

if FINETUNING_METHOD == "regression":
    FINETUNING_METRIC = "mse"
else:
    FINETUNING_METRIC = "accuracy"

# Path which will store the saved weights
MODEL_WEIGHTS_PATH = "weights"

# Number of distinct class labels
if FINETUNING_METHOD == "regression":
    NUM_CLASSES = 1
else:
    NUM_CLASSES = 4

TRACKIO_SPACE_ID = "chiabingxuan/DSA4262-FineTuning"
TRACKIO_PROJECT = "bert-finetuning-regression"

In [5]:
# Training configurations
TRAIN_CONFIG = TrainingArguments(
    # Model and data
    output_dir=os.path.join(MODEL_WEIGHTS_PATH, FINETUNED_MODEL_NAME),

    # Training hyperparameters
    learning_rate=5e-5,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    load_best_model_at_end=True,
    seed=SEED,

    # Hugging Face Hub integration
    push_to_hub=True,

    # Experiment tracking
    logging_steps=250,
    report_to=["trackio"],
    trackio_space_id=TRACKIO_SPACE_ID,
    project=TRACKIO_PROJECT,
    run_name=FINETUNED_MODEL_NAME
)

In [6]:
# LoRA configurations
LORA_ENABLED = True
LORA_CONFIG = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

## Helpers

In [7]:
def load_kaggle_dataset(dataset_path):
    dataset = dict()
    for phase in ["train", "val", "test"]:
        phase_data = dataset_load(
            KaggleDatasetAdapter.HUGGING_FACE,
            dataset_path,
            f"{phase}.csv"
        )
        dataset[phase] = phase_data
    dataset = DatasetDict(dataset)
    print(f"Dataset loaded from {dataset_path} on Kaggle!\n")

    return dataset


def preprocess_function(tokeniser, example, text_column):
    examples = tokeniser(example[text_column], padding="max_length", truncation=True)
    return examples
    

def preprocess_dataset(dataset, tokeniser_model_name, text_column):
    tokeniser = AutoTokenizer.from_pretrained(tokeniser_model_name)
    dataset = dataset.map(lambda example: preprocess_function(tokeniser, example, text_column),
                          remove_columns=[text_column], batched=True)
    
    if FINETUNING_METHOD == "regression":
        dataset = dataset.cast_column("labels", Value("float32"))
        
    print(f"Pre-processing completed with {tokeniser_model_name}!\n")
    print(dataset)

    return dataset, tokeniser

## Fine-Tuning

In [8]:
# Metrics to track performance during fine-tunings
def compute_metrics_regression(eval_metric, eval_pred):
    predictions, labels = eval_pred

    # flatten to 1D
    predictions = predictions.reshape(-1)
    labels = labels.reshape(-1)
    
    return eval_metric.compute(predictions=predictions, references=labels)


def compute_metrics_classification(eval_metric, eval_pred):
    logits, labels = eval_pred

    # Convert the logits to their corresponding predictions
    predictions = np.argmax(logits, axis=-1)
    return eval_metric.compute(predictions=predictions, references=labels)

# Number of distinct class labels
if FINETUNING_METHOD == "regression":
    chosen_compute_metrics = compute_metrics_regression
else:
    chosen_compute_metrics = compute_metrics_classification

In [9]:
def set_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)           # CPU
    torch.cuda.manual_seed(seed)      # Current GPU
    torch.cuda.manual_seed_all(seed)  # All GPUs (if multiple)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    set_seed(seed)


def check_device():
    if torch.cuda.is_available():
        device = "cuda"
        print(f"Using CUDA GPU: {torch.cuda.get_device_name()}")
        print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB\n")

    else:
        device = "cpu"
        print("Using CPU\n")
    
    return device


def load_model(model_name, num_classes, lora_enabled):
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_classes,
        device_map="auto"
    )

    if lora_enabled:
        model = get_peft_model(model, LORA_CONFIG)
        print(f"LoRA enabled:")
        model.print_trainable_parameters()
        print()

    print(f"Model architecture:")
    print(model)
    print()

    return model


def finetune_model(model, model_name, tokeniser, finetuned_model_name, repo_id, dataset, model_weights_path):
    os.makedirs(model_weights_path, exist_ok=True)
    metric = evaluate.load(FINETUNING_METRIC)

    print(f"Fine-tuning {model_name}...")
    trainer = Trainer(
        model=model,
        args=TRAIN_CONFIG,
        train_dataset=dataset["train"],
        eval_dataset=dataset["val"],
        compute_metrics=lambda eval_pred: chosen_compute_metrics(metric, eval_pred),
    )
    trainer.train()
    print("Fine-tuning completed!")

    # Add model and tokeniser to Hugging Face hub
    trainer.push_to_hub()
    tokeniser.push_to_hub(repo_id)
    print(f"Fine-tuned model {finetuned_model_name} and tokeniser uploaded to Hugging Face Hub!")

    return model


def run_finetuning_pipeline(hf_token):
    # Log in to Hugging Face
    login(hf_token)

    # Set seed for reproducibility
    set_seeds(SEED)
    print(f"Seed: {SEED}\n")

    # Check device
    device = check_device()

    # Load and verify dataset
    dataset = load_kaggle_dataset(dataset_path=DATASET_PATH)
    
    # Tokenise and process dataset
    dataset, tokeniser = preprocess_dataset(dataset=dataset, tokeniser_model_name=MODEL_NAME, text_column="text")

    # Fine-tune model
    model = load_model(
        model_name=MODEL_NAME,
        num_classes=NUM_CLASSES,
        lora_enabled=LORA_ENABLED
    )
    model_ft = finetune_model(
        model=model,
        model_name=MODEL_NAME,
        tokeniser=tokeniser,
        finetuned_model_name=FINETUNED_MODEL_NAME,
        repo_id=FINETUNED_MODEL_REPO_ID,
        dataset=dataset,
        model_weights_path=MODEL_WEIGHTS_PATH
    )

## Run Pipeline

In [10]:
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
run_finetuning_pipeline(hf_token=hf_token)

Seed: 42

Using CUDA GPU: Tesla P100-PCIE-16GB
GPU memory: 17.1GB

Dataset loaded from bingxuanchia/dsa4262-finetuning-exploration on Kaggle!



config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

* Run finished. Uploading logs to Trackio Space (please wait...)
Fine-tuning completed!


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Fine-tuned model heladepdet-bert-finetuned-classification and tokeniser uploaded to Hugging Face Hub!
